# 🌾 SIH 2026 — Crop Disease Detection & Classification

**Smart India Hackathon 2026**  
**Problem:** Early Detection and Management of Crop Diseases and Pest Infestations

This notebook builds a **multi-crop disease classifier** using transfer learning with EfficientNetB0.

**Datasets:** PlantVillage (14 crops), Paddy/Rice, Cotton, Soyabean  
**Model:** EfficientNetB0 (ImageNet pretrained) → Custom Classification Head  
**Output:** Crop → Healthy/Diseased → Disease Name → Confidence → Top-3 Predictions

## 1. Imports

In [1]:
import sys

print(sys.executable)
print(sys.version)


/opt/anaconda3/envs/sih_ml/bin/python3.11
3.11.16 (main, Aug 27 2026, 14:37:43) [Clang 20.1.8 ]


In [2]:
%pip install tensorflow


Note: you may need to restart the kernel to use updated packages.


In [3]:
# Uncomment the line below if you need to install dependencies:
# !pip install tensorflow pillow scikit-learn matplotlib seaborn pandas

import os
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU available: {len(gpus) > 0}")
if gpus:
    print(f"GPU: {gpus[0].name}")
    # Allow memory growth to avoid OOM
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

NotFoundError: dlopen(/opt/anaconda3/envs/sih_ml/lib/python3.11/site-packages/tensorflow-plugins/libmetal_plugin.dylib, 0x0006): Library not loaded: @rpath/_pywrap_tensorflow_internal.so
  Referenced from: <8B62586B-B082-3113-93AB-FD766A9960AE> /opt/anaconda3/envs/sih_ml/lib/python3.11/site-packages/tensorflow-plugins/libmetal_plugin.dylib
  Reason: tried: '/opt/anaconda3/envs/sih_ml/lib/python3.11/site-packages/tensorflow-plugins/../_solib_darwin_arm64/_U@local_Uconfig_Utf_S_S_C_Upywrap_Utensorflow_Uinternal___Uexternal_Slocal_Uconfig_Utf/_pywrap_tensorflow_internal.so' (no such file), '/opt/anaconda3/envs/sih_ml/lib/python3.11/site-packages/tensorflow-plugins/../_solib_darwin_arm64/_U@local_Uconfig_Utf_S_S_C_Upywrap_Utensorflow_Uinternal___Uexternal_Slocal_Uconfig_Utf/_pywrap_tensorflow_internal.so' (no such file), '/opt/anaconda3/envs/sih_ml/bin/../lib/_pywrap_tensorflow_internal.so' (no such file)

## 2. Configuration
All paths and hyperparameters are set here. **Change `BASE_DIR` if your datasets are in a different location.**

In [5]:
# ============================================================
# CONFIGURATION — Change paths here as needed
# ============================================================

BASE_DIR = r"C:\Users\tejar\OneDrive\Desktop\Fresh"

# Dataset directories
PLANTVILLAGE_DIR = os.path.join(BASE_DIR, "PlantVillage", "PlantVillage-Dataset-master", "raw", "color")
PADDY_DIR        = os.path.join(BASE_DIR, "paddy_split", "paddy_split")
COTTON_DIR       = os.path.join(BASE_DIR, "cotton", "Original Dataset")
SOYABEAN_DIR     = os.path.join(BASE_DIR, "soyabean", "soyabean_dataset")

# Output paths
MODEL_SAVE_PATH   = os.path.join(BASE_DIR, "disease_model.keras")
CLASS_NAMES_PATH  = os.path.join(BASE_DIR, "class_names.json")
CLASS_MAPPING_PATH = os.path.join(BASE_DIR, "class_mapping.json")

# Training hyperparameters
IMG_SIZE            = 224
BATCH_SIZE          = 32
EPOCHS              = 25
LEARNING_RATE       = 1e-3
EARLY_STOP_PATIENCE = 5
RANDOM_SEED         = 42

# Reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("Configuration set.")
print(f"  Image size : {IMG_SIZE}x{IMG_SIZE}")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Max epochs : {EPOCHS}")

Configuration set.
  Image size : 224x224
  Batch size : 32
  Max epochs : 25


## 3. Dataset Discovery
Scan the dataset directories to confirm they exist and show basic structure.

In [6]:
print("=" * 60)
print("DATASET DISCOVERY")
print("=" * 60)

datasets_info = {
    "PlantVillage": PLANTVILLAGE_DIR,
    "Paddy (Rice)": PADDY_DIR,
    "Cotton": COTTON_DIR,
    "Soyabean": SOYABEAN_DIR,
}

for name, path in datasets_info.items():
    exists = os.path.exists(path)
    status = "Found" if exists else "NOT FOUND"
    print(f"\n{'='*40}")
    print(f"{name}: {status}")
    print(f"  Path: {path}")
    if exists:
        if name == "Paddy (Rice)":
            for split in ['train', 'val', 'test']:
                sp = os.path.join(path, split)
                if os.path.exists(sp):
                    classes = [d for d in os.listdir(sp) if os.path.isdir(os.path.join(sp, d))]
                    print(f"  {split}/: {len(classes)} classes")
        else:
            classes = [d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))]
            print(f"  Classes: {len(classes)}")

DATASET DISCOVERY

PlantVillage: NOT FOUND
  Path: C:\Users\tejar\OneDrive\Desktop\Fresh/PlantVillage/PlantVillage-Dataset-master/raw/color

Paddy (Rice): NOT FOUND
  Path: C:\Users\tejar\OneDrive\Desktop\Fresh/paddy_split/paddy_split

Cotton: NOT FOUND
  Path: C:\Users\tejar\OneDrive\Desktop\Fresh/cotton/Original Dataset

Soyabean: NOT FOUND
  Path: C:\Users\tejar\OneDrive\Desktop\Fresh/soyabean/soyabean_dataset


## 4. Dataset Inspection
Show all classes and image counts for each dataset.

In [7]:
print("=" * 60)
print("DATASET INSPECTION")
print("=" * 60)

VALID_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.gif'}

def count_images_in_dir(directory):
    """Count images per class in a flat class-folder directory."""
    counts = {}
    if not os.path.exists(directory):
        return counts
    for cls in sorted(os.listdir(directory)):
        cls_path = os.path.join(directory, cls)
        if os.path.isdir(cls_path):
            n = sum(1 for f in os.listdir(cls_path)
                    if os.path.isfile(os.path.join(cls_path, f))
                    and os.path.splitext(f)[1].lower() in VALID_EXT)
            counts[cls] = n
    return counts

# --- PlantVillage ---
if os.path.exists(PLANTVILLAGE_DIR):
    print("\n--- PlantVillage (raw/color) ---")
    pv_counts = count_images_in_dir(PLANTVILLAGE_DIR)
    for cls, n in pv_counts.items():
        print(f"  {cls}: {n}")
    print(f"  TOTAL: {sum(pv_counts.values())} images, {len(pv_counts)} classes")

# --- Paddy ---
if os.path.exists(PADDY_DIR):
    print("\n--- Paddy (Rice) ---")
    paddy_total = 0
    for split in ['train', 'val', 'test']:
        sp = os.path.join(PADDY_DIR, split)
        if os.path.exists(sp):
            counts = count_images_in_dir(sp)
            total = sum(counts.values())
            paddy_total += total
            print(f"  {split}/ ({total} images):")
            for cls, n in counts.items():
                print(f"    {cls}: {n}")
    print(f"  TOTAL: {paddy_total} images")

# --- Cotton ---
if os.path.exists(COTTON_DIR):
    print("\n--- Cotton ---")
    cotton_counts = count_images_in_dir(COTTON_DIR)
    for cls, n in cotton_counts.items():
        print(f"  {cls}: {n}")
    print(f"  TOTAL: {sum(cotton_counts.values())} images, {len(cotton_counts)} classes")

# --- Soyabean ---
if os.path.exists(SOYABEAN_DIR):
    print("\n--- Soyabean ---")
    soy_counts = count_images_in_dir(SOYABEAN_DIR)
    for cls, n in soy_counts.items():
        print(f"  {cls}: {n}")
    print(f"  TOTAL: {sum(soy_counts.values())} images, {len(soy_counts)} classes")

DATASET INSPECTION


## 5. Dataset Statistics
Visualize sample images from each dataset.

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(20, 16))
fig.suptitle('Sample Images from All Datasets', fontsize=16, fontweight='bold')

sample_sources = [
    ("PlantVillage", PLANTVILLAGE_DIR),
    ("Rice", os.path.join(PADDY_DIR, 'train') if os.path.exists(PADDY_DIR) else ""),
    ("Cotton", COTTON_DIR),
    ("Soyabean", SOYABEAN_DIR),
]

for row, (name, path) in enumerate(sample_sources):
    if not path or not os.path.exists(path):
        for col in range(5):
            axes[row, col].axis('off')
        continue
    classes = sorted([d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))])
    selected = random.sample(classes, min(5, len(classes)))
    for col, cls in enumerate(selected):
        cls_path = os.path.join(path, cls)
        img_files = [f for f in os.listdir(cls_path)
                     if os.path.isfile(os.path.join(cls_path, f))
                     and os.path.splitext(f)[1].lower() in VALID_EXT]
        if img_files:
            img_path = os.path.join(cls_path, random.choice(img_files))
            try:
                img = Image.open(img_path).convert('RGB')
                axes[row, col].imshow(img)
            except Exception:
                axes[row, col].text(0.5, 0.5, 'Error', ha='center', va='center', transform=axes[row, col].transAxes)
        short = cls[:25] + '...' if len(cls) > 25 else cls
        axes[row, col].set_title(f"{name}\n{short}", fontsize=8)
        axes[row, col].axis('off')

plt.tight_layout()
plt.show()

## 6. Data Cleaning & Collection
Collect all valid image paths, assign unified class names using `Crop___Condition` format, and filter out invalid files.

In [ ]:
def collect_images(directory, class_name_func=None):
    """Collect valid image paths from a class-folder directory.
    class_name_func maps original folder name -> unified class name.
    Return None from class_name_func to skip a class.
    """
    data = []
    if not os.path.exists(directory):
        print(f"  WARNING: {directory} not found, skipping.")
        return data
    for cls in sorted(os.listdir(directory)):
        cls_path = os.path.join(directory, cls)
        if not os.path.isdir(cls_path):
            continue
        unified = class_name_func(cls) if class_name_func else cls
        if unified is None:
            continue
        for fname in os.listdir(cls_path):
            fpath = os.path.join(cls_path, fname)
            if (os.path.isfile(fpath)
                and os.path.splitext(fname)[1].lower() in VALID_EXT
                and os.path.getsize(fpath) > 100):
                data.append((fpath, unified))
    return data

# --- Class name mapping functions ---

def plantvillage_map(cls):
    """PlantVillage folders are already Crop___Condition format."""
    return cls

PADDY_MAP = {
    'bacterial_leaf_blight':   'Rice___Bacterial_Leaf_Blight',
    'bacterial_leaf_streak':   'Rice___Bacterial_Leaf_Streak',
    'bacterial_panicle_blight':'Rice___Bacterial_Panicle_Blight',
    'blast':                   'Rice___Blast',
    'brown_spot':              'Rice___Brown_Spot',
    'dead_heart':              'Rice___Dead_Heart',
    'downy_mildew':            'Rice___Downy_Mildew',
    'hispa':                   'Rice___Hispa',
    'normal':                  'Rice___healthy',
    'tungro':                  'Rice___Tungro',
}
def paddy_map(cls):
    return PADDY_MAP.get(cls, f'Rice___{cls}')

COTTON_MAP = {
    'Bacterial Blight':         'Cotton___Bacterial_Blight',
    'Curl Virus':               'Cotton___Curl_Virus',
    'Healthy Leaf':             'Cotton___healthy',
    'Herbicide Growth Damage':  'Cotton___Herbicide_Growth_Damage',
    'Leaf Hopper Jassids':      'Cotton___Leaf_Hopper_Jassids',
    'Leaf Redding':             'Cotton___Leaf_Redding',
    'Leaf Variegation':         'Cotton___Leaf_Variegation',
}
def cotton_map(cls):
    return COTTON_MAP.get(cls, f'Cotton___{cls.replace(" ", "_")}')

SOYABEAN_MAP = {
    'Healthy_Soyabean':             None,  # Skip — PlantVillage already has Soybean___healthy
    'Soyabean_Frog_Leaf_Eye':       'Soybean___Frog_Eye_Leaf_Spot',
    'Soyabean_Mosaic':              'Soybean___Mosaic_Virus',
    'Soyabean_Spectoria_Brown_Spot':'Soybean___Septoria_Brown_Spot',
}
def soyabean_map(cls):
    return SOYABEAN_MAP.get(cls)

# --- Collect from all datasets ---
print("Collecting images...\n")

pv_data = collect_images(PLANTVILLAGE_DIR, plantvillage_map)
print(f"  PlantVillage : {len(pv_data)} images")

paddy_train = collect_images(os.path.join(PADDY_DIR, 'train'), paddy_map)
paddy_val   = collect_images(os.path.join(PADDY_DIR, 'val'),   paddy_map)
paddy_test  = collect_images(os.path.join(PADDY_DIR, 'test'),  paddy_map)
print(f"  Paddy (Rice) : {len(paddy_train)} train, {len(paddy_val)} val, {len(paddy_test)} test")

cotton_data = collect_images(COTTON_DIR, cotton_map)
print(f"  Cotton       : {len(cotton_data)} images")

soy_data = collect_images(SOYABEAN_DIR, soyabean_map)
print(f"  Soyabean     : {len(soy_data)} images (healthy excluded to avoid PlantVillage overlap)")

print(f"\n  Combined     : {len(pv_data) + len(paddy_train) + len(paddy_val) + len(paddy_test) + len(cotton_data) + len(soy_data)} images")

## 7. Dataset Preparation
Split PlantVillage, Cotton, Soyabean into train/val/test (70/15/15). Use Paddy's existing splits. Combine everything.

In [ ]:
# --- Split non-paddy data ---
non_paddy = pv_data + cotton_data + soy_data
np_paths  = [p for p, _ in non_paddy]
np_labels = [l for _, l in non_paddy]

train_p, temp_p, train_l, temp_l = train_test_split(
    np_paths, np_labels, test_size=0.30, stratify=np_labels, random_state=RANDOM_SEED
)
val_p, test_p, val_l, test_l = train_test_split(
    temp_p, temp_l, test_size=0.50, stratify=temp_l, random_state=RANDOM_SEED
)

# --- Combine with paddy splits ---
train_data = list(zip(train_p, train_l)) + paddy_train
val_data   = list(zip(val_p, val_l))     + paddy_val
test_data  = list(zip(test_p, test_l))   + paddy_test

random.shuffle(train_data)
random.shuffle(val_data)
random.shuffle(test_data)

# --- Build class index ---
all_labels = [l for _, l in train_data + val_data + test_data]
class_names = sorted(set(all_labels))
num_classes = len(class_names)
class_to_idx = {name: i for i, name in enumerate(class_names)}

print(f"Final dataset summary:")
print(f"  Total classes : {num_classes}")
print(f"  Train images  : {len(train_data)}")
print(f"  Val images    : {len(val_data)}")
print(f"  Test images   : {len(test_data)}")
print(f"  Total         : {len(train_data) + len(val_data) + len(test_data)}")

# Show class distribution
print(f"\nClasses ({num_classes}):")
train_label_counts = Counter(l for _, l in train_data)
for i, name in enumerate(class_names):
    print(f"  {i:2d}. {name:55s} train={train_label_counts.get(name, 0)}")

## 8. Distribution Visualization

In [ ]:
# Plot class distribution for training set
train_counts = Counter(l for _, l in train_data)
names = class_names
counts = [train_counts.get(n, 0) for n in names]

plt.figure(figsize=(14, max(10, num_classes * 0.3)))
colors = ['#4CAF50' if 'healthy' in n.lower() else '#F44336' for n in names]
plt.barh(range(len(names)), counts, color=colors)
plt.yticks(range(len(names)), names, fontsize=7)
plt.xlabel('Number of Training Images')
plt.title('Training Set Class Distribution (Green=Healthy, Red=Diseased)')
plt.tight_layout()
plt.show()

print(f"\nSmallest class: {names[np.argmin(counts)]} ({min(counts)} images)")
print(f"Largest class : {names[np.argmax(counts)]} ({max(counts)} images)")
print(f"Median        : {int(np.median(counts))} images")

## 9. Train / Validation / Test Data Loading
Create efficient `tf.data` pipelines with image preprocessing.

In [ ]:
def load_and_preprocess(path, label):
    """Load an image file, resize to IMG_SIZE, return as float32 [0,255]."""
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)   # [0, 255] — EfficientNet handles rescaling internally
    return img, label

def make_dataset(data_list, shuffle=False):
    """Build a tf.data.Dataset from list of (path, class_name) tuples."""
    paths  = [p for p, _ in data_list]
    labels = [class_to_idx[l] for _, l in data_list]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(paths), 10000), seed=RANDOM_SEED)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

print("Building data pipelines...")
train_ds = make_dataset(train_data, shuffle=True)
val_ds   = make_dataset(val_data)
test_ds  = make_dataset(test_data)

# Verify
for imgs, lbls in train_ds.take(1):
    print(f"  Batch images shape : {imgs.shape}")
    print(f"  Batch labels shape : {lbls.shape}")
    print(f"  Pixel value range  : [{imgs.numpy().min():.0f}, {imgs.numpy().max():.0f}]")
print("Data pipelines ready.")

## 10. Data Augmentation & Model Creation
EfficientNetB0 pretrained on ImageNet with a custom classification head. Augmentation is embedded in the model (active only during training).

In [ ]:
# Data augmentation — only active during model.fit()
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.05),
], name='data_augmentation')

# Base model — frozen pretrained weights
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
)
base_model.trainable = False

# Full model
inputs  = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input_image')
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name='gap')(x)
x = layers.Dense(256, activation='relu', name='dense_head')(x)
x = layers.Dropout(0.3, name='dropout')(x)
outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

model = keras.Model(inputs, outputs, name='crop_disease_classifier')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top3_acc'),
    ],
)

model.summary(show_trainable=True)
print(f"\nTotal classes: {num_classes}")

## 11. Model Training
Train with early stopping, learning rate reduction, and model checkpointing.

In [ ]:
# Compute class weights to handle imbalanced classes
train_labels_int = [class_to_idx[l] for _, l in train_data]
cw = compute_class_weight('balanced', classes=np.arange(num_classes), y=train_labels_int)
class_weight_dict = dict(enumerate(cw))

# Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=EARLY_STOP_PATIENCE,
        restore_best_weights=True,
        verbose=1,
    ),
    ModelCheckpoint(
        MODEL_SAVE_PATH,
        monitor='val_loss',
        save_best_only=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
]

print(f"Starting training — {EPOCHS} max epochs, early stop patience={EARLY_STOP_PATIENCE}")
print(f"Training on {len(train_data)} images, validating on {len(val_data)} images\n")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1,
)

print("\nTraining complete!")
print(f"Best val_loss : {min(history.history['val_loss']):.4f}")
print(f"Best val_acc  : {max(history.history['val_accuracy']):.4f}")

## 12. Training Curves
Visualize training vs validation accuracy and loss over epochs.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
ax1.set_title('Training vs Validation Accuracy', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(history.history['loss'], label='Train Loss', linewidth=2)
ax2.plot(history.history['val_loss'], label='Val Loss', linewidth=2)
ax2.set_title('Training vs Validation Loss', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Gap analysis
final_train_acc = history.history['accuracy'][-1]
final_val_acc   = history.history['val_accuracy'][-1]
gap = final_train_acc - final_val_acc
print(f"Final train accuracy: {final_train_acc:.4f}")
print(f"Final val accuracy  : {final_val_acc:.4f}")
print(f"Train-val gap       : {gap:.4f}", end=" ")
if gap > 0.15:
    print("(significant overfitting)")
elif gap > 0.05:
    print("(mild overfitting)")
else:
    print("(good generalization)")

## 13. Test Evaluation
Evaluate on the **completely unseen** test set.

In [ ]:
print("Evaluating on test set...")
test_results = model.evaluate(test_ds, verbose=1)

print(f"\n{'='*50}")
print(f"TEST SET RESULTS")
print(f"{'='*50}")
print(f"  Test Loss       : {test_results[0]:.4f}")
print(f"  Test Accuracy   : {test_results[1]:.4f}  ({test_results[1]*100:.2f}%)")
print(f"  Top-3 Accuracy  : {test_results[2]:.4f}  ({test_results[2]*100:.2f}%)")
print(f"{'='*50}")

# Collect predictions for detailed metrics
y_true = []
y_pred = []
y_prob = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_prob.append(preds)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.concatenate(y_prob, axis=0)

print(f"\nPredictions collected: {len(y_true)} test images")

## 14. Classification Report
Per-class precision, recall, and F1-score.

In [ ]:
print("\nCLASSIFICATION REPORT")
print("=" * 90)
report = classification_report(
    y_true, y_pred,
    target_names=class_names,
    zero_division=0,
    digits=3
)
print(report)

## 15. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig_h = max(14, num_classes * 0.35)
fig_w = max(16, num_classes * 0.4)
plt.figure(figsize=(fig_w, fig_h))

sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.3,
    annot_kws={'size': 6},
)
plt.title('Confusion Matrix — Test Set', fontsize=14, fontweight='bold')
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('True', fontsize=12)
plt.xticks(rotation=90, fontsize=6)
plt.yticks(rotation=0, fontsize=6)
plt.tight_layout()
plt.show()

## 16. Model Saving
Save the trained model, class names, and class-to-output mapping.

In [ ]:
# --- Save model ---
model.save(MODEL_SAVE_PATH)
print(f"Model saved : {MODEL_SAVE_PATH}")

# --- Save class names ---
with open(CLASS_NAMES_PATH, 'w') as f:
    json.dump(class_names, f, indent=2)
print(f"Class names : {CLASS_NAMES_PATH}")

# --- Build & save class-to-output mapping ---
def parse_class_name(cname):
    """Parse 'Crop___Condition' into {'crop', 'status', 'disease'}."""
    parts = cname.split('___')
    if len(parts) != 2:
        return {'crop': cname, 'status': 'Unknown', 'disease': 'Unknown'}

    raw_crop, raw_cond = parts

    # Clean crop name
    crop = raw_crop.replace('_', ' ').strip()
    crop = crop.replace('(including sour)', '').strip()
    crop = crop.replace(', bell', ' Bell')
    crop = crop.replace('(maize)', '(Maize)')
    crop = ' '.join(w.capitalize() if w[0].islower() else w for w in crop.split())

    # Determine healthy/diseased
    if raw_cond.lower() == 'healthy':
        return {'crop': crop, 'status': 'Healthy', 'disease': 'None'}
    else:
        disease = raw_cond.replace('_', ' ').strip()
        disease = ' '.join(w.capitalize() if w[0].islower() else w for w in disease.split())
        return {'crop': crop, 'status': 'Diseased', 'disease': disease}

class_mapping = {}
for name in class_names:
    class_mapping[name] = parse_class_name(name)

with open(CLASS_MAPPING_PATH, 'w') as f:
    json.dump(class_mapping, f, indent=2)
print(f"Mapping     : {CLASS_MAPPING_PATH}")

# Preview
print(f"\n{'Class Name':<55} {'Crop':<20} {'Status':<12} Disease")
print("-" * 110)
for name in class_names:
    m = class_mapping[name]
    print(f"  {name:<53} {m['crop']:<20} {m['status']:<12} {m['disease']}")

## 17. Prediction Function
Complete prediction pipeline with visualization.

The function:
1. Loads and preprocesses the image
2. Runs the model
3. Gets Top-3 predictions
4. Determines Crop / Status / Disease
5. Calculates confidence category
6. Displays visual output

In [ ]:
def predict_image(image_path,
                   _model=None, _class_names=None, _class_mapping=None):
    """
    Predict crop disease from a leaf image.

    Args:
        image_path: Path to the image file.
    """
    # Use globals if not passed explicitly
    mdl      = _model or model
    names    = _class_names or class_names
    mapping  = _class_mapping or class_mapping

    # 1. Check file exists
    if not os.path.exists(image_path):
        print(f"Image not found: {image_path}")
        return

    # 2. Load & preprocess
    img_pil = Image.open(image_path).convert('RGB')
    img_resized = img_pil.resize((IMG_SIZE, IMG_SIZE))
    img_array = np.array(img_resized, dtype=np.float32)
    img_input = np.expand_dims(img_array, axis=0)       # (1, 224, 224, 3)

    # 3. Predict
    preds = mdl.predict(img_input, verbose=0)[0]         # (num_classes,)

    # 4. Top-3
    top3_idx   = np.argsort(preds)[-3:][::-1]
    top3_names = [names[i] for i in top3_idx]
    top3_confs = [float(preds[i]) * 100 for i in top3_idx]

    # 5. Best prediction info
    best_name = top3_names[0]
    best_conf = top3_confs[0]
    info = mapping[best_name]

    # 6. Confidence category
    if best_conf >= 80:
        assessment = "High confidence"
        bar_color  = '#4CAF50'
    elif best_conf >= 50:
        assessment = "Uncertain \u2014 clearer image recommended"
        bar_color  = '#FF9800'
    else:
        assessment = "Low confidence \u2014 expert assistance recommended"
        bar_color  = '#F44336'

    # ======== VISUALIZATION ========
    fig = plt.figure(figsize=(15, 6))
    gs  = gridspec.GridSpec(1, 2, width_ratios=[1, 1.3])

    # Left: Input image
    ax1 = fig.add_subplot(gs[0])
    ax1.imshow(img_pil)
    ax1.set_title('INPUT IMAGE', fontsize=13, fontweight='bold', pad=10)
    ax1.axis('off')

    # Right: Top-3 bar chart
    ax2 = fig.add_subplot(gs[1])
    y_pos = [2, 1, 0]
    bar_colors = [bar_color, '#64B5F6', '#90CAF9']
    bars = ax2.barh(y_pos, top3_confs, color=bar_colors, height=0.6, edgecolor='gray', linewidth=0.5)
    ax2.set_yticks(y_pos)
    short_names = [n.replace('___', ' | ') for n in top3_names]
    ax2.set_yticklabels(short_names, fontsize=10)
    ax2.set_xlabel('Model Confidence (%)', fontsize=11)
    ax2.set_title('TOP-3 PREDICTIONS', fontsize=13, fontweight='bold', pad=10)
    ax2.set_xlim(0, 105)
    for bar, conf in zip(bars, top3_confs):
        ax2.text(min(bar.get_width() + 1, 100), bar.get_y() + bar.get_height() / 2,
                 f'{conf:.2f}%', va='center', fontsize=10, fontweight='bold')
    ax2.grid(axis='x', alpha=0.2)

    plt.tight_layout()
    plt.show()

    # ======== TEXT OUTPUT ========
    print("=" * 50)
    print("PREDICTION RESULT")
    print("=" * 50)
    print(f"")
    print(f"  Crop       \u2192 {info['crop']}")
    print(f"  Status     \u2192 {info['status']}")
    print(f"  Disease    \u2192 {info['disease']}")
    print(f"  Confidence \u2192 {best_conf:.2f}%")
    print(f"  Assessment \u2192 {assessment}")
    print(f"")
    print(f"  Top 3 Predictions:")
    print(f"")
    for rank, (n, c) in enumerate(zip(top3_names, top3_confs), 1):
        m = mapping[n]
        label = f"{m['crop']} - {m['disease']}" if m['status'] != 'Healthy' else f"{m['crop']} - Healthy"
        print(f"    {rank}. {label:40s} {c:.2f}%")
    print(f"")
    print("=" * 50)

print("predict_image() function defined.")

## 18. Test Your Own Image

**Change ONLY the `IMAGE_PATH` below**, then run the cell.

```python
IMAGE_PATH = "path/to/your/leaf_image.jpg"
```

In [ ]:
# ============================================================
# CHANGE THIS PATH to test your own image
# ============================================================

IMAGE_PATH = "path/to/your/image.jpg"

# Run prediction
predict_image(IMAGE_PATH)

## 19. Example Prediction
Automatically test with a random image from the test set.

In [ ]:
# Pick a random test image
example_path, example_label = random.choice(test_data)
print(f"Test image  : {example_path}")
print(f"True class  : {example_label}")
true_info = class_mapping[example_label]
print(f"True crop   : {true_info['crop']}")
print(f"True status : {true_info['status']}")
print(f"True disease: {true_info['disease']}")
print()

predict_image(example_path)

## 20. Summary

### What was built
- Multi-crop disease classifier covering **16 crops** and **~58 classes**
- Transfer learning with **EfficientNetB0** (ImageNet pretrained)
- Combined **4 datasets**: PlantVillage, Paddy, Cotton, Soyabean

### Saved outputs
| File | Description |
|------|-------------|
| `disease_model.keras` | Trained Keras model |
| `class_names.json` | Ordered list of class names |
| `class_mapping.json` | Class → Crop/Status/Disease mapping |

### To use in production
```python
model = keras.models.load_model("disease_model.keras")
class_names = json.load(open("class_names.json"))
class_mapping = json.load(open("class_mapping.json"))
predict_image("leaf_photo.jpg", _model=model, _class_names=class_names, _class_mapping=class_mapping)
```

---
*SIH 2026 — Crop Disease Detection Prototype*